# Network Security Audit

This notebook queries the OCP audit SQLite datastore to present findings for:

- **OCP-30 — Network Port Restriction**: Access to sensitive ports is restricted to required networks and subnets. Evidence on-cluster: CNI plugin posture (egress-firewall enablement on OVN-Kubernetes), active egress firewall rules, IngressController endpoint-publishing strategy, route TLS posture, and the count of insecure (non-TLS) routes.
- **OCP-31 — Service Mesh Enforcement**: Use of a service mesh (e.g. Istio / OpenShift Service Mesh) is enforced for in-cluster traffic. Evidence on-cluster: install posture of the `servicemesh-operator`, presence of a `ServiceMeshControlPlane` (SMCP) resource, and whether SMCP mTLS is set to `strict`.
- **OCP-32 — Native Network Policies**: Kubernetes-native NetworkPolicies are used to control traffic between pods and namespaces. Evidence on-cluster: count of NetworkPolicy resources and the breakdown of namespaces that have at least one NetworkPolicy versus the total number of namespaces.
- **OCP-33 — Encryption in Transit**: Encryption in transit is enabled for cluster traffic and external-facing routes. Evidence on-cluster: OVN-Kubernetes IPsec mode, IngressController minimum TLS version, count of routes without TLS, and count of routes allowing insecure HTTP termination.
- **OCP-34 — CNI Plugin Usage**: A supported CNI plugin is in use and supports NetworkPolicy enforcement. Evidence on-cluster: active CNI plugin type and the cluster-wide count of NetworkPolicy resources.
- **OCP-35 — External Egress/Ingress Boundary Protection**: Dedicated controls such as WAF or API Gateway are in place for external-facing ingress traffic. Evidence on-cluster: presence/enabled-state of a WAF record on the IngressController and IngressController availability.
- **OCP-37 — Internal Service Exposure Control**: Exposure of internal cluster services to the internal corporate network is strictly controlled and audited. Evidence on-cluster: API server/console TLS posture, audit profile, `additionalCORSAllowedOrigins`, and presence of an audit log forwarder.
- **OCP-38 — Integration with Underlying Cloud / Network Infrastructure**: Platform, application, or service is correctly and securely provisioned, configured, and governed in relation to the foundational cloud and network services. Evidence on-cluster: cloud platform type, control-plane and infrastructure topology (HA), master node count, and CNI network type.


In [ ]:
import os
import re
import sys

import pandas as pd

# Ensure the repo root is importable before loading project modules.
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))

from notebook_style import bootstrap, style_table  # noqa: E402

print("python:", sys.executable)
print("cwd:", os.getcwd())

# bootstrap() adds ../datastore to sys.path, which is required before
# importing schema.models below.
session, engine = bootstrap()

from schema.models import (  # noqa: E402
    ApiServerConsoleAccess,
    Cluster,
    ClusterOverview,
    IngressBoundaryProtection,
    MonitoringAuditLogging,
    NetworkSecurityMesh,
)

print("Connected to:", engine.url)

## Cluster Inventory


In [ ]:
df_clusters = pd.read_sql(
    session.query(
        Cluster.id,
        Cluster.cluster_name,
        Cluster.cluster_context,
        Cluster.cluster_server,
    ).statement,
    engine,
)
print(f"{len(df_clusters)} cluster(s) in dataset")
style_table(df_clusters)

In [ ]:
def _detail_has(value, needle):
    if value is None:
        return False
    return needle in str(value)


def _extract_int(value, regex):
    if value is None:
        return None
    m = regex.search(str(value))
    return int(m.group(1)) if m else None

---
## OCP-30: Network Port Restriction

*Access to sensitive ports is restricted to required networks and subnets.*

On-cluster evidence is the **CNI plugin** posture (whether OVN-Kubernetes is active and whether the **egress firewall** feature is enabled), the count of **active egress firewall rules**, the **IngressController** posture (endpoint-publishing strategy and minimum TLS version), the breakdown of **route counts** by TLS termination, and the count of **insecure routes** (non-TLS / `insecure-edge-termination-policy=Allow`). Record types in scope: `cni`, `egress_firewall_count` from `network_security_mesh`, and `ingresscontroller`, `route_count`, `insecure_routes` from `ingress_boundary_protection`.

> **Note on scope.** The current export does not enumerate per-namespace port allowlists or NodePort services individually; the closest signals available on-cluster are surfaced here. Treat insecure-route count and absence of egress-firewall rules as the strongest indicators of an unrestricted boundary.

### Network port / boundary records (per cluster)


In [ ]:
OCP30_NSM_RECORD_TYPES = ("cni", "egress_firewall_count")
OCP30_IBP_RECORD_TYPES = ("ingresscontroller", "route_count", "insecure_routes")

df_ocp30_nsm = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        NetworkSecurityMesh.record_type,
        NetworkSecurityMesh.component_name,
        NetworkSecurityMesh.status,
        NetworkSecurityMesh.namespace,
        NetworkSecurityMesh.detail_1,
        NetworkSecurityMesh.detail_2,
        NetworkSecurityMesh.detail_3,
        NetworkSecurityMesh.detail_4,
    )
    .join(Cluster, NetworkSecurityMesh.cluster_id == Cluster.id)
    .filter(NetworkSecurityMesh.record_type.in_(OCP30_NSM_RECORD_TYPES))
    .order_by(Cluster.cluster_name, NetworkSecurityMesh.record_type)
    .statement,
    engine,
)

df_ocp30_ibp = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        IngressBoundaryProtection.record_type,
        IngressBoundaryProtection.component_name,
        IngressBoundaryProtection.status,
        IngressBoundaryProtection.namespace,
        IngressBoundaryProtection.detail_1,
        IngressBoundaryProtection.detail_2,
        IngressBoundaryProtection.detail_3,
        IngressBoundaryProtection.detail_4,
    )
    .join(Cluster, IngressBoundaryProtection.cluster_id == Cluster.id)
    .filter(IngressBoundaryProtection.record_type.in_(OCP30_IBP_RECORD_TYPES))
    .order_by(Cluster.cluster_name, IngressBoundaryProtection.record_type)
    .statement,
    engine,
)

df_ocp30 = pd.concat([df_ocp30_nsm, df_ocp30_ibp], ignore_index=True).sort_values(
    ["cluster_name", "record_type"]
)
style_table(df_ocp30, caption="OCP-30: Raw network port / boundary records")

### OCP-30: Compliance flags

Per-cluster flags derived from the raw records. A cluster is considered **compliant** when:

- `egress_firewall_enabled` — `detail_2` on the `cni` row contains `egress-firewall=enabled`.
- `egress_firewall_rules` — count parsed from the `status` of the `egress_firewall_count` row (must be `> 0`).
- `ingress_available` — `ingresscontroller` row has `status = Available`.
- `ingress_tls_min_v12` — `detail_3` on the `ingresscontroller` row contains `tls-min-version=VersionTLS12` (or higher).
- `insecure_route_count` — integer value from `status` on the `insecure_routes` row (must be `0`).

Only clusters that fail one or more of these checks are displayed.


In [ ]:
_INT_RE = re.compile(r"(\d+)")

rows = []
for cluster_name in sorted(df_clusters["cluster_name"].unique()):
    nsm = df_ocp30_nsm[df_ocp30_nsm["cluster_name"] == cluster_name]
    ibp = df_ocp30_ibp[df_ocp30_ibp["cluster_name"] == cluster_name]

    cni_rows = nsm[nsm["record_type"] == "cni"]
    egress_firewall_enabled = not cni_rows.empty and _detail_has(
        cni_rows.iloc[0]["detail_2"], "egress-firewall=enabled"
    )

    ef_rows = nsm[nsm["record_type"] == "egress_firewall_count"]
    if ef_rows.empty:
        egress_firewall_rules = 0
    else:
        m = _INT_RE.search(str(ef_rows.iloc[0]["status"] or ""))
        egress_firewall_rules = int(m.group(1)) if m else 0

    ic_rows = ibp[ibp["record_type"] == "ingresscontroller"]
    if ic_rows.empty:
        ingress_available = False
        ingress_tls_min_v12 = False
    else:
        ic = ic_rows.iloc[0]
        ingress_available = str(ic["status"]) == "Available"
        ingress_tls_min_v12 = _detail_has(
            ic["detail_3"], "tls-min-version=VersionTLS12"
        ) or _detail_has(ic["detail_3"], "tls-min-version=VersionTLS13")

    ir_rows = ibp[ibp["record_type"] == "insecure_routes"]
    if ir_rows.empty:
        insecure_route_count = None
    else:
        m = _INT_RE.search(str(ir_rows.iloc[0]["status"] or ""))
        insecure_route_count = int(m.group(1)) if m else None

    rows.append(
        {
            "cluster_name": cluster_name,
            "egress_firewall_enabled": egress_firewall_enabled,
            "egress_firewall_rules": egress_firewall_rules,
            "ingress_available": ingress_available,
            "ingress_tls_min_v12": ingress_tls_min_v12,
            "insecure_route_count": insecure_route_count,
        }
    )

df_ocp30_flags = pd.DataFrame(rows)
df_ocp30_flags["compliant"] = (
    df_ocp30_flags["egress_firewall_enabled"]
    & (df_ocp30_flags["egress_firewall_rules"] > 0)
    & df_ocp30_flags["ingress_available"]
    & df_ocp30_flags["ingress_tls_min_v12"]
    & df_ocp30_flags["insecure_route_count"].fillna(-1).eq(0)
)
df_ocp30_noncompliant = df_ocp30_flags[~df_ocp30_flags["compliant"]].copy()
print(
    f"{len(df_ocp30_noncompliant)} of {len(df_ocp30_flags)} cluster(s) non-compliant for OCP-30"
)
style_table(df_ocp30_noncompliant, caption="OCP-30: Non-compliant clusters")

---
## OCP-31: Service Mesh Enforcement

*Use of a service mesh (e.g. OpenShift Service Mesh / Istio) is enforced for in-cluster traffic.*

On-cluster evidence is the install posture of the **`servicemesh-operator`**, the presence of one or more **`ServiceMeshControlPlane` (SMCP)** resources, and whether the SMCP enforces **mutual TLS** in `strict` mode. Record types in scope: `operator` (filtered to `component_name = servicemesh-operator`) and `smcp` from `network_security_mesh`.

### Service mesh records (per cluster)


In [ ]:
df_ocp31 = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        NetworkSecurityMesh.record_type,
        NetworkSecurityMesh.component_name,
        NetworkSecurityMesh.status,
        NetworkSecurityMesh.namespace,
        NetworkSecurityMesh.detail_1,
        NetworkSecurityMesh.detail_2,
        NetworkSecurityMesh.detail_3,
        NetworkSecurityMesh.detail_4,
    )
    .join(Cluster, NetworkSecurityMesh.cluster_id == Cluster.id)
    .filter(
        (
            (NetworkSecurityMesh.record_type == "operator")
            & (NetworkSecurityMesh.component_name == "servicemesh-operator")
        )
        | (NetworkSecurityMesh.record_type == "smcp")
    )
    .order_by(Cluster.cluster_name, NetworkSecurityMesh.record_type)
    .statement,
    engine,
)
style_table(df_ocp31, caption="OCP-31: Raw service mesh records")

### OCP-31: Compliance flags

Per-cluster flags derived from the raw records. A cluster is considered **compliant** when:

- `service_mesh_installed` — `operator` row for `servicemesh-operator` has `status = installed`.
- `smcp_present` — at least one `smcp` row exists for the cluster.
- `smcp_ready` — at least one `smcp` row has `status = Ready`.
- `mtls_strict` — at least one `smcp` row has `detail_2` containing `mtls=strict`.

Only clusters that fail one or more of these checks are displayed.


In [ ]:
rows = []
for cluster_name in sorted(df_clusters["cluster_name"].unique()):
    sub = df_ocp31[df_ocp31["cluster_name"] == cluster_name]

    op_rows = sub[
        (sub["record_type"] == "operator")
        & (sub["component_name"] == "servicemesh-operator")
    ]
    service_mesh_installed = (
        not op_rows.empty and str(op_rows.iloc[0]["status"]) == "installed"
    )

    smcp_rows = sub[sub["record_type"] == "smcp"]
    smcp_present = not smcp_rows.empty
    smcp_ready = bool((smcp_rows["status"] == "Ready").any()) if smcp_present else False
    mtls_strict = (
        bool(smcp_rows["detail_2"].apply(lambda v: _detail_has(v, "mtls=strict")).any())
        if smcp_present
        else False
    )

    rows.append(
        {
            "cluster_name": cluster_name,
            "service_mesh_installed": service_mesh_installed,
            "smcp_present": smcp_present,
            "smcp_ready": smcp_ready,
            "mtls_strict": mtls_strict,
        }
    )

df_ocp31_flags = pd.DataFrame(rows)
df_ocp31_flags["compliant"] = df_ocp31_flags[
    ["service_mesh_installed", "smcp_present", "smcp_ready", "mtls_strict"]
].all(axis=1)
df_ocp31_noncompliant = df_ocp31_flags[~df_ocp31_flags["compliant"]].copy()
print(
    f"{len(df_ocp31_noncompliant)} of {len(df_ocp31_flags)} cluster(s) non-compliant for OCP-31"
)
style_table(df_ocp31_noncompliant, caption="OCP-31: Non-compliant clusters")

---
## OCP-32: Native Network Policies

*Kubernetes-native NetworkPolicies are used to control traffic between pods and namespaces.*

On-cluster evidence is the count of `NetworkPolicy` resources cluster-wide and the breakdown of namespaces that have at least one NetworkPolicy versus the total number of namespaces. Record type in scope: `networkpolicy_count` from `network_security_mesh`, where `status` holds the total NetworkPolicy count, `detail_1` holds `namespaces-with-policies=N`, and `detail_2` holds `total-namespaces=M`.

### NetworkPolicy records (per cluster)


In [ ]:
df_ocp32 = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        NetworkSecurityMesh.record_type,
        NetworkSecurityMesh.component_name,
        NetworkSecurityMesh.status,
        NetworkSecurityMesh.namespace,
        NetworkSecurityMesh.detail_1,
        NetworkSecurityMesh.detail_2,
        NetworkSecurityMesh.detail_3,
        NetworkSecurityMesh.detail_4,
    )
    .join(Cluster, NetworkSecurityMesh.cluster_id == Cluster.id)
    .filter(NetworkSecurityMesh.record_type == "networkpolicy_count")
    .order_by(Cluster.cluster_name)
    .statement,
    engine,
)
style_table(df_ocp32, caption="OCP-32: Raw NetworkPolicy records")

### OCP-32: Compliance flags

Per-cluster flags derived from the raw records. A cluster is considered **compliant** when:

- `networkpolicy_total` — integer `status` from the `networkpolicy_count` row (must be `> 0`).
- `namespaces_with_policies` — integer parsed from `detail_1` (`namespaces-with-policies=N`).
- `total_namespaces` — integer parsed from `detail_2` (`total-namespaces=M`).
- `coverage_pct` — `100 * namespaces_with_policies / total_namespaces` (must be `>= 50%`).

Only clusters that fail one or more of these checks are displayed.


In [ ]:
_NS_WITH_POL_RE = re.compile(r"namespaces-with-policies=(\d+)")
_TOTAL_NS_RE = re.compile(r"total-namespaces=(\d+)")

COVERAGE_THRESHOLD_PCT = 50.0

rows = []
for cluster_name in sorted(df_clusters["cluster_name"].unique()):
    sub = df_ocp32[df_ocp32["cluster_name"] == cluster_name]
    if sub.empty:
        rows.append(
            {
                "cluster_name": cluster_name,
                "networkpolicy_total": 0,
                "namespaces_with_policies": None,
                "total_namespaces": None,
                "coverage_pct": None,
            }
        )
        continue
    r = sub.iloc[0]
    m = _INT_RE.search(str(r["status"] or ""))
    np_total = int(m.group(1)) if m else 0
    ns_with = _extract_int(r["detail_1"], _NS_WITH_POL_RE)
    ns_total = _extract_int(r["detail_2"], _TOTAL_NS_RE)
    if ns_with is not None and ns_total and ns_total > 0:
        coverage_pct = round(100.0 * ns_with / ns_total, 1)
    else:
        coverage_pct = None
    rows.append(
        {
            "cluster_name": cluster_name,
            "networkpolicy_total": np_total,
            "namespaces_with_policies": ns_with,
            "total_namespaces": ns_total,
            "coverage_pct": coverage_pct,
        }
    )

df_ocp32_flags = pd.DataFrame(rows)
df_ocp32_flags["compliant"] = (
    df_ocp32_flags["networkpolicy_total"] > 0
) & df_ocp32_flags["coverage_pct"].fillna(-1).ge(COVERAGE_THRESHOLD_PCT)
df_ocp32_noncompliant = df_ocp32_flags[~df_ocp32_flags["compliant"]].copy()
print(
    f"{len(df_ocp32_noncompliant)} of {len(df_ocp32_flags)} cluster(s) non-compliant for OCP-32 "
    f"(coverage threshold {COVERAGE_THRESHOLD_PCT:.0f}%)"
)
style_table(df_ocp32_noncompliant, caption="OCP-32: Non-compliant clusters")

---
## OCP-33: Encryption in Transit

*Encryption in transit is enabled for cluster traffic and external-facing routes.*

On-cluster evidence is the **OVN-Kubernetes IPsec mode** (`Full`/`Enabled` ⇒ pod-to-pod traffic encrypted), the **IngressController minimum TLS version**, the **count of routes with no TLS**, and the **count of routes that allow insecure (HTTP) edge termination**. Record types in scope: `network_encryption` from `network_security_mesh`; `ingresscontroller`, `route_count`, `insecure_routes` from `ingress_boundary_protection`.

### Encryption-in-transit records (per cluster)


In [ ]:
OCP33_NSM_RECORD_TYPES = ("network_encryption",)
OCP33_IBP_RECORD_TYPES = ("ingresscontroller", "route_count", "insecure_routes")

df_ocp33_nsm = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        NetworkSecurityMesh.record_type,
        NetworkSecurityMesh.component_name,
        NetworkSecurityMesh.status,
        NetworkSecurityMesh.namespace,
        NetworkSecurityMesh.detail_1,
        NetworkSecurityMesh.detail_2,
        NetworkSecurityMesh.detail_3,
        NetworkSecurityMesh.detail_4,
    )
    .join(Cluster, NetworkSecurityMesh.cluster_id == Cluster.id)
    .filter(NetworkSecurityMesh.record_type.in_(OCP33_NSM_RECORD_TYPES))
    .order_by(Cluster.cluster_name)
    .statement,
    engine,
)

df_ocp33_ibp = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        IngressBoundaryProtection.record_type,
        IngressBoundaryProtection.component_name,
        IngressBoundaryProtection.status,
        IngressBoundaryProtection.namespace,
        IngressBoundaryProtection.detail_1,
        IngressBoundaryProtection.detail_2,
        IngressBoundaryProtection.detail_3,
        IngressBoundaryProtection.detail_4,
    )
    .join(Cluster, IngressBoundaryProtection.cluster_id == Cluster.id)
    .filter(IngressBoundaryProtection.record_type.in_(OCP33_IBP_RECORD_TYPES))
    .order_by(Cluster.cluster_name, IngressBoundaryProtection.record_type)
    .statement,
    engine,
)

df_ocp33 = pd.concat([df_ocp33_nsm, df_ocp33_ibp], ignore_index=True).sort_values(
    ["cluster_name", "record_type"]
)
style_table(df_ocp33, caption="OCP-33: Raw encryption-in-transit records")

### OCP-33: Compliance flags

Per-cluster flags derived from the raw records. A cluster is considered **compliant** when:

- `ipsec_enabled` — `status` of the `network_encryption` row contains `ipsec:Full` or `ipsec:Enabled`.
- `ingress_tls_min_v12` — `detail_3` on the `ingresscontroller` row is `tls-min-version=VersionTLS12` (or higher).
- `insecure_route_count` — integer value from `status` on the `insecure_routes` row (must be `0`).

Only clusters that fail one or more of these checks are displayed.


In [ ]:
rows = []
for cluster_name in sorted(df_clusters["cluster_name"].unique()):
    nsm = df_ocp33_nsm[df_ocp33_nsm["cluster_name"] == cluster_name]
    ibp = df_ocp33_ibp[df_ocp33_ibp["cluster_name"] == cluster_name]

    enc_rows = nsm[nsm["record_type"] == "network_encryption"]
    if enc_rows.empty:
        ipsec_mode = ""
        ipsec_enabled = False
    else:
        ipsec_mode = str(enc_rows.iloc[0]["status"] or "")
        ipsec_enabled = ("ipsec:Full" in ipsec_mode) or ("ipsec:Enabled" in ipsec_mode)

    ic_rows = ibp[ibp["record_type"] == "ingresscontroller"]
    if ic_rows.empty:
        ingress_tls_min_v12 = False
    else:
        d3 = ic_rows.iloc[0]["detail_3"]
        ingress_tls_min_v12 = _detail_has(
            d3, "tls-min-version=VersionTLS12"
        ) or _detail_has(d3, "tls-min-version=VersionTLS13")

    ir_rows = ibp[ibp["record_type"] == "insecure_routes"]
    if ir_rows.empty:
        insecure_route_count = None
    else:
        m = _INT_RE.search(str(ir_rows.iloc[0]["status"] or ""))
        insecure_route_count = int(m.group(1)) if m else None

    rows.append(
        {
            "cluster_name": cluster_name,
            "ipsec_mode": ipsec_mode,
            "ipsec_enabled": ipsec_enabled,
            "ingress_tls_min_v12": ingress_tls_min_v12,
            "insecure_route_count": insecure_route_count,
        }
    )

df_ocp33_flags = pd.DataFrame(rows)
df_ocp33_flags["compliant"] = (
    df_ocp33_flags["ipsec_enabled"]
    & df_ocp33_flags["ingress_tls_min_v12"]
    & df_ocp33_flags["insecure_route_count"].fillna(-1).eq(0)
)
df_ocp33_noncompliant = df_ocp33_flags[~df_ocp33_flags["compliant"]].copy()
print(
    f"{len(df_ocp33_noncompliant)} of {len(df_ocp33_flags)} cluster(s) non-compliant for OCP-33"
)
style_table(df_ocp33_noncompliant, caption="OCP-33: Non-compliant clusters")

---
## OCP-34: CNI Plugin Usage

*A supported CNI plugin is in use and supports NetworkPolicy enforcement to restrict pod-to-pod communication.*

On-cluster evidence is the **active CNI plugin** (`OVNKubernetes` / `OpenShiftSDN`) reported on the `cni` record, whether the plugin is `active`, and whether **NetworkPolicies** are actually deployed (cluster-wide count). Record types in scope: `cni` and `networkpolicy_count` from `network_security_mesh`.

### CNI plugin records (per cluster)


In [ ]:
OCP34_RECORD_TYPES = ("cni", "networkpolicy_count")

df_ocp34 = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        NetworkSecurityMesh.record_type,
        NetworkSecurityMesh.component_name,
        NetworkSecurityMesh.status,
        NetworkSecurityMesh.namespace,
        NetworkSecurityMesh.detail_1,
        NetworkSecurityMesh.detail_2,
        NetworkSecurityMesh.detail_3,
        NetworkSecurityMesh.detail_4,
    )
    .join(Cluster, NetworkSecurityMesh.cluster_id == Cluster.id)
    .filter(NetworkSecurityMesh.record_type.in_(OCP34_RECORD_TYPES))
    .order_by(Cluster.cluster_name, NetworkSecurityMesh.record_type)
    .statement,
    engine,
)
style_table(df_ocp34, caption="OCP-34: Raw CNI plugin records")

### OCP-34: Compliance flags

Per-cluster flags derived from the raw records. A cluster is considered **compliant** when:

- `cni_plugin` — `component_name` on the `cni` row (must be set).
- `cni_active` — `status = active` on the `cni` row.
- `cni_supports_netpol` — `cni_plugin` is one of `OVNKubernetes`, `OpenShiftSDN`, `Cilium`, `Calico`.
- `networkpolicy_total` — integer `status` from the `networkpolicy_count` row (must be `> 0`).

Only clusters that fail one or more of these checks are displayed.


In [ ]:
NETPOL_CAPABLE_CNIS = {"OVNKubernetes", "OpenShiftSDN", "Cilium", "Calico"}

rows = []
for cluster_name in sorted(df_clusters["cluster_name"].unique()):
    sub = df_ocp34[df_ocp34["cluster_name"] == cluster_name]

    cni_rows = sub[sub["record_type"] == "cni"]
    if cni_rows.empty:
        cni_plugin = ""
        cni_active = False
    else:
        cni_plugin = str(cni_rows.iloc[0]["component_name"] or "")
        cni_active = str(cni_rows.iloc[0]["status"] or "") == "active"
    cni_supports_netpol = cni_plugin in NETPOL_CAPABLE_CNIS

    np_rows = sub[sub["record_type"] == "networkpolicy_count"]
    if np_rows.empty:
        networkpolicy_total = 0
    else:
        m = _INT_RE.search(str(np_rows.iloc[0]["status"] or ""))
        networkpolicy_total = int(m.group(1)) if m else 0

    rows.append(
        {
            "cluster_name": cluster_name,
            "cni_plugin": cni_plugin,
            "cni_active": cni_active,
            "cni_supports_netpol": cni_supports_netpol,
            "networkpolicy_total": networkpolicy_total,
        }
    )

df_ocp34_flags = pd.DataFrame(rows)
df_ocp34_flags["compliant"] = (
    df_ocp34_flags["cni_active"]
    & df_ocp34_flags["cni_supports_netpol"]
    & (df_ocp34_flags["networkpolicy_total"] > 0)
)
df_ocp34_noncompliant = df_ocp34_flags[~df_ocp34_flags["compliant"]].copy()
print(
    f"{len(df_ocp34_noncompliant)} of {len(df_ocp34_flags)} cluster(s) non-compliant for OCP-34"
)
style_table(df_ocp34_noncompliant, caption="OCP-34: Non-compliant clusters")

---
## OCP-35: External Egress/Ingress Boundary Protection

*Dedicated controls — such as a Web Application Firewall (WAF) or API Gateway — are in place for external-facing ingress traffic to protect application APIs and services.*

On-cluster evidence is the presence and enabled-state of a **WAF** record (e.g. `aws-waf`, `modsecurity`, F5/Cloudflare/Akamai annotations on the IngressController) and the IngressController **availability**. Record types in scope: `waf` and `ingresscontroller` from `ingress_boundary_protection`.

### Boundary protection records (per cluster)


In [ ]:
OCP35_RECORD_TYPES = ("waf", "ingresscontroller")

df_ocp35 = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        IngressBoundaryProtection.record_type,
        IngressBoundaryProtection.component_name,
        IngressBoundaryProtection.status,
        IngressBoundaryProtection.namespace,
        IngressBoundaryProtection.detail_1,
        IngressBoundaryProtection.detail_2,
        IngressBoundaryProtection.detail_3,
        IngressBoundaryProtection.detail_4,
    )
    .join(Cluster, IngressBoundaryProtection.cluster_id == Cluster.id)
    .filter(IngressBoundaryProtection.record_type.in_(OCP35_RECORD_TYPES))
    .order_by(Cluster.cluster_name, IngressBoundaryProtection.record_type)
    .statement,
    engine,
)
style_table(df_ocp35, caption="OCP-35: Raw boundary-protection records")

### OCP-35: Compliance flags

Per-cluster flags derived from the raw records. A cluster is considered **compliant** when:

- `waf_present` — at least one `waf` row exists for the cluster.
- `waf_enabled` — at least one `waf` row has `status = enabled`.
- `ingress_available` — `ingresscontroller` row has `status = Available`.

Only clusters that fail one or more of these checks are displayed.


In [ ]:
rows = []
for cluster_name in sorted(df_clusters["cluster_name"].unique()):
    sub = df_ocp35[df_ocp35["cluster_name"] == cluster_name]

    waf_rows = sub[sub["record_type"] == "waf"]
    waf_present = not waf_rows.empty
    waf_enabled = (
        bool((waf_rows["status"] == "enabled").any()) if waf_present else False
    )
    waf_components = (
        ";".join(sorted({str(c) for c in waf_rows["component_name"] if c}))
        if waf_present
        else ""
    )

    ic_rows = sub[sub["record_type"] == "ingresscontroller"]
    ingress_available = (
        not ic_rows.empty and str(ic_rows.iloc[0]["status"]) == "Available"
    )

    rows.append(
        {
            "cluster_name": cluster_name,
            "waf_present": waf_present,
            "waf_enabled": waf_enabled,
            "waf_components": waf_components,
            "ingress_available": ingress_available,
        }
    )

df_ocp35_flags = pd.DataFrame(rows)
df_ocp35_flags["compliant"] = (
    df_ocp35_flags["waf_present"]
    & df_ocp35_flags["waf_enabled"]
    & df_ocp35_flags["ingress_available"]
)
df_ocp35_noncompliant = df_ocp35_flags[~df_ocp35_flags["compliant"]].copy()
print(
    f"{len(df_ocp35_noncompliant)} of {len(df_ocp35_flags)} cluster(s) non-compliant for OCP-35"
)
style_table(df_ocp35_noncompliant, caption="OCP-35: Non-compliant clusters")

---
## OCP-37: Internal Service Exposure Control

*Exposure of internal cluster services (e.g. metrics, cluster API endpoints) to the internal corporate network is strictly controlled and audited via perimeter security controls.*

On-cluster evidence is the **API server / Console TLS posture** (`tls_security_profile_type`, `tls_min_version`), the **audit profile** on the API server (records all administrative access), the **`additionalCORSAllowedOrigins`** setting (limits which origins can reach the API), and whether a **log forwarder** is configured (so audit events leave the cluster to a SIEM). Sources: `apiserver_console_access` and `monitoring_audit_logging` (`record_type='log_forwarder'`).

### API/console exposure records (per cluster)


In [ ]:
df_ocp37_api = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        ApiServerConsoleAccess.api_server_url,
        ApiServerConsoleAccess.console_url,
        ApiServerConsoleAccess.tls_security_profile_type,
        ApiServerConsoleAccess.tls_min_version,
        ApiServerConsoleAccess.audit_profile,
        ApiServerConsoleAccess.additional_cors_origins,
    )
    .join(Cluster, ApiServerConsoleAccess.cluster_id == Cluster.id)
    .order_by(Cluster.cluster_name)
    .statement,
    engine,
)

df_ocp37_log = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        MonitoringAuditLogging.record_type,
        MonitoringAuditLogging.component_name,
        MonitoringAuditLogging.status,
    )
    .join(Cluster, MonitoringAuditLogging.cluster_id == Cluster.id)
    .filter(MonitoringAuditLogging.record_type == "log_forwarder")
    .order_by(Cluster.cluster_name)
    .statement,
    engine,
)

style_table(df_ocp37_api, caption="OCP-37: API & console exposure (per cluster)")

### OCP-37: Compliance flags

Per-cluster flags derived from the raw records. A cluster is considered **compliant** when:

- `tls_intermediate_or_modern` — `tls_security_profile_type` is `Intermediate` or `Modern`.
- `tls_min_v12` — `tls_min_version` is `VersionTLS12` or `VersionTLS13`.
- `audit_profile_set` — `audit_profile` is set and not `None` / `Default` (i.e. an explicit profile such as `WriteRequestBodies` or `AllRequestBodies`).
- `cors_origins_set` — `additional_cors_origins` is non-empty (CORS allow-list configured).
- `log_forwarder_present` — at least one `log_forwarder` row in `monitoring_audit_logging` for the cluster.

Only clusters that fail one or more of these checks are displayed.


In [ ]:
TLS_OK_PROFILES = {"Intermediate", "Modern"}
TLS_OK_VERSIONS = {"VersionTLS12", "VersionTLS13"}
EXPLICIT_AUDIT_PROFILES = {"WriteRequestBodies", "AllRequestBodies"}

rows = []
for cluster_name in sorted(df_clusters["cluster_name"].unique()):
    api = df_ocp37_api[df_ocp37_api["cluster_name"] == cluster_name]
    log = df_ocp37_log[df_ocp37_log["cluster_name"] == cluster_name]

    if api.empty:
        tls_profile = ""
        tls_min = ""
        audit_profile = ""
        cors = ""
    else:
        r = api.iloc[0]
        tls_profile = str(r["tls_security_profile_type"] or "")
        tls_min = str(r["tls_min_version"] or "")
        audit_profile = str(r["audit_profile"] or "")
        cors = str(r["additional_cors_origins"] or "")

    rows.append(
        {
            "cluster_name": cluster_name,
            "tls_security_profile_type": tls_profile,
            "tls_intermediate_or_modern": tls_profile in TLS_OK_PROFILES,
            "tls_min_version": tls_min,
            "tls_min_v12": tls_min in TLS_OK_VERSIONS,
            "audit_profile": audit_profile,
            "audit_profile_set": audit_profile in EXPLICIT_AUDIT_PROFILES,
            "cors_origins_set": bool(cors.strip()),
            "log_forwarder_present": not log.empty,
        }
    )

df_ocp37_flags = pd.DataFrame(rows)
df_ocp37_flags["compliant"] = (
    df_ocp37_flags["tls_intermediate_or_modern"]
    & df_ocp37_flags["tls_min_v12"]
    & df_ocp37_flags["audit_profile_set"]
    & df_ocp37_flags["cors_origins_set"]
    & df_ocp37_flags["log_forwarder_present"]
)
df_ocp37_noncompliant = df_ocp37_flags[~df_ocp37_flags["compliant"]].copy()
print(
    f"{len(df_ocp37_noncompliant)} of {len(df_ocp37_flags)} cluster(s) non-compliant for OCP-37"
)
style_table(df_ocp37_noncompliant, caption="OCP-37: Non-compliant clusters")

---
## OCP-38: Integration with Underlying Cloud / Network Infrastructure

*Security and architectural requirements ensure a platform, application, or service is correctly and securely provisioned, configured, and governed in relation to the foundational cloud and network services it relies on.*

On-cluster evidence is the **platform** (cloud provider — AWS, Azure, GCP, vSphere, BareMetal, …), **control-plane and infrastructure topology** (HighlyAvailable vs SingleReplica), the **master node count** (≥3 for HA), and the **CNI network type** (a supported plugin). Source: `cluster_overview`.

### Cloud / infrastructure records (per cluster)


In [ ]:
df_ocp38 = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        ClusterOverview.platform,
        ClusterOverview.control_plane_topology,
        ClusterOverview.infrastructure_topology,
        ClusterOverview.master_count,
        ClusterOverview.worker_count,
        ClusterOverview.network_type,
        ClusterOverview.default_ingress_domain,
    )
    .join(Cluster, ClusterOverview.cluster_id == Cluster.id)
    .order_by(Cluster.cluster_name)
    .statement,
    engine,
)
style_table(df_ocp38, caption="OCP-38: Cloud & infrastructure posture (per cluster)")

### OCP-38: Compliance flags

Per-cluster flags derived from the raw records. A cluster is considered **compliant** when:

- `platform_known` — `platform` is one of `AWS`, `Azure`, `GCP`, `vSphere`, `BareMetal`, `IBMCloud`, `OpenStack`, `PowerVS`, `Nutanix` (i.e. an officially supported provider, not `None`).
- `control_plane_ha` — `control_plane_topology` is `HighlyAvailable`.
- `infrastructure_ha` — `infrastructure_topology` is `HighlyAvailable`.
- `master_count_ge_3` — `master_count >= 3`.
- `supported_network_type` — `network_type` is `OVNKubernetes` or `OpenShiftSDN`.

Only clusters that fail one or more of these checks are displayed.


In [ ]:
KNOWN_PLATFORMS = {
    "AWS",
    "Azure",
    "GCP",
    "vSphere",
    "BareMetal",
    "IBMCloud",
    "OpenStack",
    "PowerVS",
    "Nutanix",
}
SUPPORTED_NETWORK_TYPES = {"OVNKubernetes", "OpenShiftSDN"}

rows = []
for cluster_name in sorted(df_clusters["cluster_name"].unique()):
    sub = df_ocp38[df_ocp38["cluster_name"] == cluster_name]
    if sub.empty:
        rows.append(
            {
                "cluster_name": cluster_name,
                "platform": "",
                "platform_known": False,
                "control_plane_topology": "",
                "control_plane_ha": False,
                "infrastructure_topology": "",
                "infrastructure_ha": False,
                "master_count": None,
                "master_count_ge_3": False,
                "network_type": "",
                "supported_network_type": False,
            }
        )
        continue

    r = sub.iloc[0]
    platform = str(r["platform"] or "")
    cp_topo = str(r["control_plane_topology"] or "")
    inf_topo = str(r["infrastructure_topology"] or "")
    master_count = r["master_count"]
    network_type = str(r["network_type"] or "")

    rows.append(
        {
            "cluster_name": cluster_name,
            "platform": platform,
            "platform_known": platform in KNOWN_PLATFORMS,
            "control_plane_topology": cp_topo,
            "control_plane_ha": cp_topo == "HighlyAvailable",
            "infrastructure_topology": inf_topo,
            "infrastructure_ha": inf_topo == "HighlyAvailable",
            "master_count": master_count,
            "master_count_ge_3": (master_count is not None)
            and (int(master_count) >= 3),
            "network_type": network_type,
            "supported_network_type": network_type in SUPPORTED_NETWORK_TYPES,
        }
    )

df_ocp38_flags = pd.DataFrame(rows)
df_ocp38_flags["compliant"] = (
    df_ocp38_flags["platform_known"]
    & df_ocp38_flags["control_plane_ha"]
    & df_ocp38_flags["infrastructure_ha"]
    & df_ocp38_flags["master_count_ge_3"]
    & df_ocp38_flags["supported_network_type"]
)
df_ocp38_noncompliant = df_ocp38_flags[~df_ocp38_flags["compliant"]].copy()
print(
    f"{len(df_ocp38_noncompliant)} of {len(df_ocp38_flags)} cluster(s) non-compliant for OCP-38"
)
style_table(df_ocp38_noncompliant, caption="OCP-38: Non-compliant clusters")

In [ ]:
session.close()
print("Session closed.")